<a href="https://colab.research.google.com/github/SanikaPatil1008/Deep_Learning/blob/main/Exp_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Colab Notebooks/Deep_Learning/dataset.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done")

Done


In [ ]:
# 1. IMPORT LIBRARIES
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import os
import zipfile
import shutil

# 4. CHECK DATASET STRUCTURE
for root, dirs, files in os.walk("/content/dataset"):
    print(root, "=>", dirs)

# 5. FIND CORRECT DATASET PATH
# TRY COMMON STRUCTURES AUTOMATICALLY
if len(os.listdir("/content/dataset")) > 0:
    subfolder = os.listdir("/content/dataset")[0]
    possible_path = os.path.join("/content/dataset", subfolder)
    if os.path.isdir(possible_path):
        dataset_path = possible_path
    else:
        dataset_path = "/content/dataset"
else:
    raise Exception("Dataset not found!")

print("Using dataset path:", dataset_path)

# 6. DATA PREPROCESSING
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# 7. LOAD PRETRAINED RESNET50
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
for layer in base_model.layers:
    layer.trainable = False

# 8. ADD CLASSIFIER HEAD
num_classes = len(train_data.class_indices)

model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# 9. COMPILE MODEL (FROZEN)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 10. TRAIN FROZEN MODEL
history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# 11. FINE TUNING
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 12. TRAIN FINE-TUNED MODEL
history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# 13. SAVE MODEL
model.save("transfer_learning_resnet50.h5")

print("Training Complete & Model Saved!")